<a href="https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/05_day6-15_project_tasks.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 在浏览器中打开本 notebook 后，点击菜单 **代码执行程序 → 全部运行** 即可按顺序执行所有单元格。

# Day 6–15｜项目实战：学生管理系统

**项目目标**：用前 5 天学的全部知识（Python、字符串、Pandas、Linux、MySQL），开发一个**真正能跑的学生管理系统**，数据存在你云服务器的 MySQL 里，程序也部署在云服务器上运行。

**最终系统功能清单**：

| 模块 | 功能 |
|---|---|
| 数据层 | MySQL 存学生/班级/成绩，Python 通过 pymysql 读写 |
| 增删改查 | 添加学生、按姓名/班级查询、修改成绩、删除学生 |
| 统计报表 | 各科平均分、总分排名、及格率、班级对比 |
| 文件交互 | CSV 批量导入学生、导出报表、自动生成文字成绩报告 |
| 部署 | 程序上传到云服务器，nohup 后台常驻运行 |

**10 天的节奏**：每天 4 小时 = 任务讲解 0.5 小时 + 动手开发 3 小时 + 答疑/复盘 0.5 小时。

**开发方式**：每天先在 Colab 里开发调试（用 sqlite 或连服务器 MySQL 都可以），功能稳定后归入最终程序；**Day 14 统一到服务器部署**。

> 📋 本 notebook 是**任务书**：每天做什么、做到什么程度算完成，都写在这里。
> 完整的参考实现在另一本：`06_day6-15_project_solution.ipynb`——**先自己做，卡住 30 分钟以上再看**。

---
## 🗓️ Day 6｜数据库设计：把系统的"地基"打好

**今日目标**：在服务器 MySQL 中建好项目数据库和全部表。

**任务步骤**：
1. 纸上/笔记里先设计：系统要存什么？（学生、班级、成绩）→ 画三张表的字段草图
2. SSH 登录服务器，用 `learn` 账号创建数据库 `student_mgmt`（utf8mb4）
3. 编写并执行建表 SQL：

```sql
CREATE DATABASE IF NOT EXISTS student_mgmt DEFAULT CHARACTER SET utf8mb4;
USE student_mgmt;

CREATE TABLE classes (
    id      INT PRIMARY KEY AUTO_INCREMENT,
    cls_name VARCHAR(50) NOT NULL UNIQUE,   -- 班级名，不允许重复
    teacher  VARCHAR(50)
);

CREATE TABLE students (
    id      INT PRIMARY KEY AUTO_INCREMENT,
    name    VARCHAR(50) NOT NULL,
    cls_id  INT,                            -- 关联 classes.id
    gender  VARCHAR(4),
    age     INT,
    FOREIGN KEY (cls_id) REFERENCES classes(id)
);

CREATE TABLE scores (
    id      INT PRIMARY KEY AUTO_INCREMENT,
    stu_id  INT NOT NULL,                   -- 关联 students.id
    subject VARCHAR(20) NOT NULL,           -- 科目：数学/英语/语文…
    score   DECIMAL(5,1),
    FOREIGN KEY (stu_id) REFERENCES students(id),
    UNIQUE KEY uq_stu_sub (stu_id, subject) -- 同一学生同一科目只一条
);
```

4. 插入测试数据：2 个班、至少 6 名学生、每人 2–3 科成绩

**✅ 验收标准**：`SHOW TABLES;` 有 3 张表；用一条 JOIN 查询能看到"姓名-班级-科目-分数"；测试数据 ≥ 12 行成绩记录。

**💡 用到**：Day 4 的建库建表、外键概念（新：`FOREIGN KEY` 保证成绩必须属于真实存在的学生）。

---
## 🗓️ Day 7｜Python 连接 + 程序骨架 + 添加学生

**今日目标**：Python 能连上服务器 MySQL，程序有菜单骨架，并完成第一个功能"添加学生"。

**任务步骤**：
1. Colab 里 `!pip install pymysql`，写好连接函数 `get_conn()`（host 填服务器 IP）
2. 写主菜单循环骨架：

```python
def main():
    while True:
        print("""\n===== 学生管理系统 =====
1. 添加学生   2. 查询学生   3. 修改成绩
4. 删除学生   5. 统计报表   6. 导入/导出
0. 退出""")
        choice = input("请选择：").strip()
        if choice == "0":
            print("再见！"); break
        elif choice == "1": add_student()
        # ... 其他分支先 print("待开发")
```

3. 实现 `add_student()`：input 收集姓名/班级/性别/年龄 → `INSERT INTO students`（班级名要先查 classes 表换 cls_id）→ 打印"添加成功，学号为 X"

**✅ 验收标准**：菜单能循环显示；添加 1 名学生后，在服务器 mysql 客户端 `SELECT * FROM students;` 能看到新记录；输错菜单项不会崩溃。

**💡 用到**：Day 1 的循环/input/字符串，Day 5 的 Python 连接 MySQL。

---
## 🗓️ Day 8｜查询功能：把数据"看"清楚

**今日目标**：实现三个查询——查看全部、按姓名模糊查、按班级查。

**任务步骤**：
1. `show_all()`：`SELECT s.id, s.name, c.cls_name, s.gender, s.age FROM students s JOIN classes c ...`，用 f-string 对齐打印成表格（Day 1 的对齐技巧！）
2. `search_by_name()`：input 关键字 → `WHERE name LIKE %s` 带 `%关键字%`
3. `search_by_cls()`：input 班级名 → JOIN 查询该班所有人
4. 进阶：每个查询结果末尾打印"共 X 条记录"

**✅ 验收标准**：三种查询都能返回正确结果；查不到时友好提示"未找到"；表格列对齐美观。

**💡 用到**：Day 4 的 SELECT/WHERE/LIKE/JOIN，Day 1 的 f-string 对齐。

> ⚠️ SQL 注入提醒：用户输入**必须**用参数占位符 `%s` 传进去，绝不 f-string 拼 SQL！

---
## 🗓️ Day 9｜修改与删除：胆大心细

**今日目标**：实现修改成绩和删除学生，**操作前先确认**。

**任务步骤**：
1. `update_score()`：input 学号和科目 → 先 SELECT 显示当前成绩 → 让用户确认 → UPDATE 新成绩
2. `delete_student()`：input 学号 → 显示该生信息 → 输入 `yes` 才执行（先删 scores 再删 students，否则外键报错）
3. 处理异常：学号不存在、输入非数字时不能崩溃（`try/except`）

**✅ 验收标准**：修改/删除后数据库内容正确；取消操作（不输入 yes）数据不变；输入乱打字不崩溃。

**💡 用到**：Day 4 的 UPDATE/DELETE 安全心法（先 SELECT 确认），外键约束的处理顺序。

---
## 🗓️ Day 10｜统计报表（上）+ 测验 2

**今日目标**：用 `pandas.read_sql` 做统计——各科平均分、总分排名榜。

**任务步骤**：
1. `report_subject_avg()`：`SELECT subject, ROUND(AVG(score),1) ... GROUP BY subject` 打印
2. `report_top(n)`：JOIN 三表按学生汇总总分，`ORDER BY total DESC LIMIT n` 打印排名榜（名次列用循环计数生成）
3. 把报表同时导出为 CSV 文件

**✅ 验收标准**：两个报表数字与手算一致；排名榜格式整齐；CSV 能正常打开。

**📝 下午做【测验 2】**（覆盖 Day 6–9 的项目开发内容和 SQL），80 分以上继续。

**💡 用到**：Day 5 的聚合/JOIN/read_sql。

---
## 🗓️ Day 11｜统计报表（下）：及格率与班级对比

**今日目标**：做两张更有"分析味"的报表。

**任务步骤**：
1. `report_pass_rate()`：每科的及格率（`SUM(score>=60)/COUNT(*)`）和优秀率（≥90）
2. `report_cls_compare()`：按班级对比各科平均分（JOIN students+scores+classes，GROUP BY 班级+科目）
3. 用 Pandas 把"班级×科目"的结果转置成交叉表（`pivot_table`），更像一张真正的分析报表

**✅ 验收标准**：及格率用百分比格式显示（如 `83.3%`）；交叉表行列清晰。

**💡 用到**：Day 5 的 HAVING/聚合技巧，Day 2 的 Pandas 变形。

---
## 🗓️ Day 12｜CSV 批量导入：一次录一个班

**今日目标**：从 CSV 文件批量导入学生和成绩。

**任务步骤**：
1. 准备 `import_demo.csv`（自己编 8 行：姓名,班级,性别,年龄,科目,分数）
2. `import_csv()`：`pd.read_csv` → 逐行处理：班级不存在就自动建班、学生不存在就自动建学生 → 插入 scores
3. 数据校验：分数必须在 0–100，不合法的行跳过并打印警告；导入结束打印"成功 X 条，跳过 Y 条"
4. 防重复：同一学生同一科目已存在时，用 `INSERT ... ON DUPLICATE KEY UPDATE` 更新而不是报错

**✅ 验收标准**：8 行 CSV 全部正确入库；重复导入同一文件不会出错也不会产生重复数据。

**💡 用到**：Day 2 的 read_csv，Day 1 的字符串清洗，Day 9 的异常处理。

---
## 🗓️ Day 13｜导出 + 自动生成文字报告

**今日目标**：把数据变成"能交差"的成果——导出报表 + 生成每个学生的文字成绩报告。

**任务步骤**：
1. `export_report()`：把总分排名、班级对比两张报表导出为一个 Excel/CSV 文件
2. `student_report()`：输入学号 → 查出该生所有成绩 → 用 f-string 生成一段**自然语言文字报告**，例如：

```
========== 学生成绩报告 ==========
姓名：李雷    班级：一班    学号：1
--------------------------------
数学：88 分（班级平均 86.3，超过平均）
英语：92 分（班级平均 86.7，超过平均）
--------------------------------
总分：180，班级排名：1 / 3
评语：综合成绩优秀，保持！
```

3. 批量生成全班报告，每个学生存一个 `report_学号.txt`

**✅ 验收标准**：报告数字与数据库一致；评语按总分分档（优秀/良好/及格/待努力）；批量生成不报错。

**💡 用到**：Day 1 字符串处理的巅峰应用——f-string、对齐、split/join 全用上。

---
## 🗓️ Day 14｜部署：让程序在服务器上 7×24 运行

**今日目标**：把程序上传到云服务器，后台常驻运行。

**任务步骤**：
1. 把最终代码整理成 `app.py`（一个文件，含全部功能）
2. 上传到服务器：`scp app.py ubuntu@服务器IP:~/student-mgmt/`（或在服务器上 `nano app.py` 直接粘贴）
3. 服务器上装依赖：`sudo apt install -y python3-pip && pip3 install pymysql pandas`（或 apt 安装 python3-pandas）
4. 前台试运行：`python3 app.py` 确认功能正常
5. 后台常驻：`nohup python3 app.py > app.log 2>&1 &`，用 `ps aux | grep app.py` 确认进程在跑
6. 查看日志：`tail -f app.log`（Ctrl+C 退出查看，程序不会停）
7. 关闭程序：`kill 进程号`

**✅ 验收标准**：断开 SSH 重连后 `ps aux` 仍能看到进程；日志文件有运行记录。

**💡 用到**：Day 3 的 Linux 全家桶——scp/mkdir/ps/kill/重定向 `>`。

---
## 🗓️ Day 15｜完善日 + 总答疑 + 测验 3

**今日目标**：查漏补缺，把项目打磨到"能给别人演示"的程度。

**任务步骤**：
1. 走查全部功能一遍，记录 bug 和体验问题，逐个修复
2. 给 `app.py` 写文件头注释（功能、作者、日期、用法）+ 关键函数写注释
3. 写 `README.md`：项目介绍、功能清单、安装运行步骤（放服务器同目录）
4. 彩排一次 5 分钟演示：添加→查询→统计→报告→看日志
5. **集中答疑**：把 10 天攒下的问题清单全部过一遍
6. 做【测验 3】（项目综合）

**✅ 验收标准**：演示全流程不卡壳；README 能让别人照着跑起来；问题清单全部清零。

---
## 📒 我的问题清单（随手记录，答疑前整理）

学习中遇到任何卡住的地方，立刻记在下面表格里，**答疑时间集中解决**：

| 日期 | 问题描述 | 状态（待问/已解决） | 答案要点 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---
➡️ **参考实现**（自己先做完再对照）：[https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/06_day6-15_project_solution.ipynb](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/06_day6-15_project_solution.ipynb)

➡️ **项目完成后**：Day 16–20 进阶内容 —— [https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/07_day16-20_advanced.ipynb](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/07_day16-20_advanced.ipynb)